# 00 · Attention Lab 导读

**这是一份仓库导读和预测核算，不是 Mac GPU 实验报告。**

目标：从“解释一种机制”走向“在同一模型上检验读取方式的速度、质量与成本”。正式实验由 Python 模块执行；Notebook 展示相同版本的代码、证据与可重放的小例子。

本页不下载模型、不调用 API、不运行 GPU、不创建 GitHub 仓库。

## 1. 找到唯一的项目目录

下面只定位仓库，避免 Notebook 工作目录变化时误读另一份结果。

In [1]:
from pathlib import Path
import json, hashlib, importlib.util, sys, inspect

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / ".attention_lab_bootstrap").is_file()), None)
assert root is not None, "请从 attention-lab 项目目录或 notebooks 子目录打开本页"
print("已找到 Attention Lab。远端是否发布请看实际发布回执，不由目录存在推断。")

已找到 Attention Lab。远端是否发布请看实际发布回执，不由目录存在推断。


## 2. 原始材料与当前方案不是同一层

`docs/archive/PLAN_v3.md` 保持原字节。当前执行要求在 `docs/PLAN_CURRENT.md`，修订原因在 `docs/DECISIONS.md`。教材也不回写；新增解释明确是派生内容。

SHA-256 校验只说明归档没有改变，**不证明原件里的全部技术或性能主张已验证**。

In [2]:
manifest = json.loads((root / "docs/source_manifest.json").read_text(encoding="utf-8"))
for item in manifest["files"]:
    data = (root / item["path"]).read_bytes()
    assert len(data) == item["bytes"]
    assert hashlib.sha256(data).hexdigest() == item["sha256"]
    print(item["path"], "—", item["bytes"], "bytes — hash matched")

docs/archive/PLAN_v3.md — 26663 bytes — hash matched
docs/archive/roofline_original.py — 3445 bytes — hash matched
docs/reference/handbook_with_exercises_updated(4).html — 838364 bytes — hash matched
docs/archive/PLAN_v2.md — 35801 bytes — hash matched
docs/archive/PLAN_training_proposal.md — 27778 bytes — hash matched
docs/archive/attention_review_evidence.zip — 5003 bytes — hash matched
docs/archive/attention_tradeoff_plan_v2.zip — 28520 bytes — hash matched
docs/archive/transformer_teaching_plan.zip — 18748 bytes — hash matched
docs/archive/kv_principle_rerun_tests.txt — 1657 bytes — hash matched


## 3. 调用正式的预测工具，而不是另写一份公式

这里加载 `scripts/roofline.py`。它是已上传原脚本的修订版：ρ=0可用、k截断到候选域、格式标签不固定写BF16、默认不填设备带宽。

下面只看读取预算的实现片段；完整来源仍在同一个Python文件。

In [3]:
source_path = root / "scripts/roofline.py"
spec = importlib.util.spec_from_file_location("attention_lab_roofline_reading", source_path)
roofline = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = roofline
spec.loader.exec_module(roofline)
source_sha = hashlib.sha256(source_path.read_bytes()).hexdigest()
print("预测工具源码 SHA-256:", source_sha)
lines = inspect.getsource(roofline.estimate).splitlines()
start = next(i for i, line in enumerate(lines) if "fixed_count =" in line)
print("\n".join(lines[start:start + 14]))

预测工具源码 SHA-256: 5506a9ec34adfc04bdaef46f4f62f8768378aad87a95afb4705fe56f14da87eb
    fixed_count = min(fixed, n)
    candidate_count = n - fixed_count
    k_requested = round(rho * n)  # Python ties-to-even, matching the supplied script.
    k = min(k_requested, candidate_count)
    selected = fixed_count + k
    per_position = 2 * model.layers * model.kv_heads * model.head_dim * element_bytes
    weights = model.parameters * element_bytes
    dense_kv = per_position * n
    scanned_n = n if scan_scope == "all" else candidate_count
    index_per_position = model.layers * model.kv_heads * r * index_bytes
    index_stored = n * index_per_position
    index_read = scanned_n * index_per_position
    selected_kv = selected * per_position
    d_s1 = index_read + selected_kv


## 4. 只算载荷，不把它当成真实生成速度

以v3记录的4B形状、32K历史、ρ=1/32、索引宽度32作为**配置算例**。没有传入带宽，所以不会产生毫秒或吞吐的伪测量。

S1只计索引和一次入选KV；S2另计gather写入与注意力重读。实际缓存命中、top-k扫描、投影与执行效率都还没测。

In [4]:
prediction = roofline.estimate(roofline.MODELS["Qwen3-4B"], 32768)
for key in ("k", "selected_count", "weights_bytes", "dense_kv_bytes",
            "index_stored_bytes", "d_s1_bytes", "d_s2_bytes",
            "saved_s1_scenario_ms"):
    print(f"{key}: {prediction[key]}")
assert prediction["saved_s1_scenario_ms"] is None
assert prediction["kind"] == "prediction"

k: 1024
selected_count: 1092
weights_bytes: 8044936192
dense_kv_bytes: 4831838208
index_stored_bytes: 603979776
d_s1_bytes: 763748352
d_s2_bytes: 1085792256
saved_s1_scenario_ms: None


## 5. 改条件之前先作判断

ρ=0应该只保留固定集合；ρ=1应该最多全选，不能超过历史长度。它们是边界检查，不是稀疏速度成绩。

In [5]:
shape = roofline.MODELS["Qwen3-0.6B"]
fixed_only = roofline.estimate(shape, 512, rho=0)
full_selection = roofline.estimate(shape, 512, rho=1)
assert fixed_only["k"] == 0 and fixed_only["selected_count"] == 68
assert full_selection["selected_count"] == 512
assert full_selection["degenerate_full_selection"]
print("ρ=0：", fixed_only["selected_count"], "个位置；ρ=1：", full_selection["selected_count"], "个位置。")

ρ=0： 68 个位置；ρ=1： 512 个位置。


## 6. 实测尚未发生

当前页面验证的是文件、Python预测公式和边界，不是Qwen能力，也不是用户Mac的MPS支持。

下一项是 `docs/tasks/01_G0_G1A.md`：本机环境预检、真实注意力等价测试、8K/16K/32K微基准。完成后才生成第一份实际实验课。

**预测 → 运行 → 核对 → 解释 → 改一个条件再验证。** 成功不要求D一定更快；需要的是结果足以支持一个有边界的设计判断。